# caban — pipeline driver

Notebook entry point for the caban analysis pipeline. Each cell runs one stage; you can stop at any point and inspect state in the live kernel.

**Architecture**
- `caban.config.PipelineConfig` — all user-tunable switches.
- `caban.loader.load_all_mice(cfg)` — returns a `SimpleNamespace` with every metadata dict, session dict, mapping accumulator, and engram-pass result.
- `caban.pipeline.*` — thin wrappers around the analysis modules (`caban.population`, `caban.isomap`, `caban.epoch_analysis`, ...).
- `caban.analyses` — top-level analysis script (every `plot_*` / `enable_*` / `optimize_*` block, run in notebook globals).

**Globals injection.** After `load_all_mice`, the cell
```python
globals().update(vars(ds))
```
brings every per-mouse dict (`TFC_cond`, `Test_B`, `engram_id`, every `mappings_all_*`, every `dpath_*`, ...) into the notebook's top-level namespace. Both `ds.TFC_cond` and bare `TFC_cond` then reference the same object, so script-style analysis blocks and debug snippets work unchanged.


## 1. Setup

    "`%autoreload 2` rebuilds modules on edit. Heavy code lives in `caban/*.py` and is reloaded automatically; the dataset itself is **not** rebuilt — you keep your loaded `ds`."

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, importlib
import numpy as np
import matplotlib.pyplot as plt

import caban.config
import caban.loader
import caban.pipeline
from caban.config import PipelineConfig
from caban.loader import load_all_mice
from caban.pipeline import (
    resolve_continuity_params,
    engram_idx_by_mouse,
    run_engram_sanity_plots,
    run_population_pca,
    run_population_pca_all_modes,
    run_isomap,
    run_epoch_pv,
    run_cross_session_epoch_pv,
)

## 2. Build config

Edit fields here to override defaults. All ~90 switches from `caban/main.py` are exposed.


In [ ]:
cfg = PipelineConfig(
    # DEBUG=True,                # restrict to one mouse for quick smoke tests
    # plot_pf_raw_maps=True,     # heavy per-cell PF plots
    # optimize_parameters=True,  # run raw-S decoder Optuna study
    # enable_population_curve=True,
)
print(cfg)

## 3. Load all mice

Builds CrossReg + Session objects, runs per-mapping accumulators, the unified engram-identity pass, and the merged-PF backfill. Caches: NPY artifacts under `NPY_SAVE_PATH`.

In [ ]:
ds = load_all_mice(cfg)


# Also expose every cfg switch as a bare global so blocks in caban/analyses.py
# (which reference plot_PSTH, DEVEL_SWITCH, etc. without a ``cfg.`` prefix)
# find them.

print(f"PLOTS_DIR  = {ds.PLOTS_DIR}")
print(f"#mice      = {len(ds.mouse_list)}")
print(f"groups     = { {g: len(ms) for g, ms in ds.mice_per_group.items()} }")


## 4. (Optional) Save loaded dataset to disk for fast restart

Pickling lets you reload in a fresh kernel without re-running the per-mouse loop. Session objects are heavy; expect a multi-GB pickle. Only do this once you trust your `cfg`.

In [ ]:
# import pickle
# pickle_path = os.path.join(ds.TFC_cond_savepath, 'ds_cache.pkl')
# with open(pickle_path, 'wb') as f:
#     pickle.dump(ds, f, protocol=pickle.HIGHEST_PROTOCOL)
# print('saved ->', pickle_path)

## Analysis sections

The cells below follow the exact top-level pathway of `caban/main.py`, one section per cell, in the same order. Each cell executes its line-range of `caban/main.py` via `run_main_section(...)`, so tracebacks point to real source lines and `%debug` lands on the right code.

Every block is gated by its own `cfg.plot_*` / `cfg.enable_*` / `cfg.optimize_*` switch — already exported as bare globals by the load cell above. Toggle a switch off on `cfg`, re-run `globals().update(cfg.export_globals())`, and the corresponding cell becomes a no-op.


In [ ]:
# Section functions live in caban.sections.
# Each takes (ds, cfg) plus any cross-section state via kwargs
# and is gated by its cfg.plot_* / enable_* switch.
from caban.sections import (
    run_sp_rates,
    run_binned_sp_rates,
    run_ROIs,
    run_proportional_activities,
    run_LT_firing_rate_changes,
    run_PSTH,
    run_pf_and_loc,
    run_occupancy_analysis,
    run_LT_pfs,
    run_LT_decoding,
    run_zone_crossreg,
    run_continuity_and_paramsets,
    run_optimize_raw_decoder,
    run_optimize_pf_decoder,
    run_paradigm_A,
    run_paradigm_B,
    run_paradigm_C,
    run_paradigm_D1,
    run_paradigm_D2,
    run_paradigm_E1,
    run_paradigm_E2,
    run_paradigm_F,
    run_mixedlm_cross_vs_within,
    run_mixedlm_vs_tfc_cond,
    run_pv_correlation_2d,
    run_epoch_pv_within,
    run_epoch_pv_cross,
    run_population_pca,
    run_isomap,
    run_population_vectors,
    run_population_vector_distances,
    run_binned_activities,
)

# Shared state threaded between sections (None until produced).
raw_params = pf_params = None
lt_cont_pvt = None; use_PCT_error = None
mt_A_results_2D = mt_B_results_2D = mt_C_results_2D = None


### 9. Spike-rate panels

`caban/main.py` L1474–1539


In [ ]:
run_sp_rates(ds, cfg)


### 10. Binned spike-rate panels

`caban/main.py` L1540–1590


In [ ]:
run_binned_sp_rates(ds, cfg)


### 11. ROI maps

`caban/main.py` L1591–1598


In [ ]:
run_ROIs(ds, cfg)


### 12. Proportional activities

`caban/main.py` L1599–1609


In [ ]:
run_proportional_activities(ds, cfg)


### 13. LT firing-rate changes

`caban/main.py` L1610–1646


In [ ]:
run_LT_firing_rate_changes(ds, cfg)


### 14. PSTH

`caban/main.py` L1647–1681


In [ ]:
run_PSTH(ds, cfg)


### 15. Place fields and locations

`caban/main.py` L1682–1779


In [ ]:
run_pf_and_loc(ds, cfg)


### 16. Occupancy / trajectory / immobility

`caban/main.py` L1780–1818


In [ ]:
run_occupancy_analysis(ds, cfg)


### 17. LT place fields

`caban/main.py` L1819–2095


In [ ]:
run_LT_pfs(ds, cfg)


### 18. LT decoding

`caban/main.py` L2096–2888


In [ ]:
_result = run_LT_decoding(ds, cfg)
lt_cont_pvt   = _result.get("lt_cont_pvt") if _result else None
use_PCT_error = _result.get("use_PCT_error") if _result else None


### 19. Zone cross-registration suite

`caban/main.py` L2889–3037


In [ ]:
run_zone_crossreg(ds, cfg, lt_cont_pvt=lt_cont_pvt, use_PCT_error=use_PCT_error)


### 20. Decoder constants and continuity sigmas

`caban/main.py` L3038–3215


In [ ]:
_result = run_continuity_and_paramsets(ds, cfg)
raw_params = _result["raw_params"] if _result else None
pf_params  = _result["pf_params"]  if _result else None


### 21. Optimize raw-decoder parameters

`caban/main.py` L3216–3285


In [ ]:
run_optimize_raw_decoder(ds, cfg)


### 22. Optimize PF-decoder parameters

`caban/main.py` L3286–3537


In [ ]:
run_optimize_pf_decoder(ds, cfg)


### 23. 2D Bayesian decoder paradigm A

`caban/main.py` L3783–4016


In [ ]:
_result = run_paradigm_A(ds, cfg, raw_params=raw_params, pf_params=pf_params)
mt_A_results_2D = _result["mt_A_results_2D"] if _result else None


### 24. 2D Bayesian decoder paradigm B

`caban/main.py` L4017–4171


In [ ]:
_result = run_paradigm_B(ds, cfg, raw_params=raw_params, pf_params=pf_params)
mt_B_results_2D = _result["mt_B_results_2D"] if _result else None


### 25. 2D Bayesian decoder paradigm C

`caban/main.py` L4172–4327


In [ ]:
_result = run_paradigm_C(ds, cfg, raw_params=raw_params, pf_params=pf_params)
mt_C_results_2D = _result["mt_C_results_2D"] if _result else None


### 26. 2D Bayesian decoder paradigm D1

`caban/main.py` L4328–4477


In [ ]:
run_paradigm_D1(ds, cfg, raw_params=raw_params, pf_params=pf_params)


### 27. 2D Bayesian decoder paradigm D2

`caban/main.py` L4478–4627


In [ ]:
run_paradigm_D2(ds, cfg, raw_params=raw_params, pf_params=pf_params)


### 28. 2D Bayesian decoder paradigm E1

`caban/main.py` L4628–4775


In [ ]:
run_paradigm_E1(ds, cfg, raw_params=raw_params, pf_params=pf_params)


### 29. 2D Bayesian decoder paradigm E2

`caban/main.py` L4776–4924


In [ ]:
run_paradigm_E2(ds, cfg, raw_params=raw_params, pf_params=pf_params)


### 30. 2D Bayesian decoder paradigm F

`caban/main.py` L4925–5127


In [ ]:
run_paradigm_F(ds, cfg, raw_params=raw_params, pf_params=pf_params)


### 31. TFC cross-session vs within-session MixedLM

`caban/main.py` L5128–5155


In [ ]:
run_mixedlm_cross_vs_within(ds, cfg, mt_A_results_2D=mt_A_results_2D, mt_B_results_2D=mt_B_results_2D, mt_C_results_2D=mt_C_results_2D)


### 32. TFC vs TFC_cond baseline MixedLM

`caban/main.py` L5156–5181


In [ ]:
run_mixedlm_vs_tfc_cond(ds, cfg, mt_A_results_2D=mt_A_results_2D, mt_B_results_2D=mt_B_results_2D, mt_C_results_2D=mt_C_results_2D)


### 33. 2D Population Vector (PV) correlation

`caban/main.py` L5182–5562


In [ ]:
run_pv_correlation_2d(ds, cfg, raw_params=raw_params, pf_params=pf_params)


### 34. Within-session epoch-PV similarity

`caban/main.py` L5563–5631


In [ ]:
run_epoch_pv_within(ds, cfg)


### 35. Cross-session epoch-PV similarity

`caban/main.py` L5632–5708


In [ ]:
run_epoch_pv_cross(ds, cfg)


### 36. Population PCA trajectory analysis (incl. engram sanity)

`caban/main.py` L5709–5861


In [ ]:
run_population_pca(ds, cfg)


### 37. Isomap manifold pipeline

`caban/main.py` L5862–5882


In [ ]:
run_isomap(ds, cfg)


### 38. Population vectors

`caban/main.py` L5883–5937


In [ ]:
run_population_vectors(ds, cfg)


### 39. Population vector distances

`caban/main.py` L5938–6367


In [ ]:
run_population_vector_distances(ds, cfg)


### 40. Binned activities

`caban/main.py` L6368–6371


In [ ]:
run_binned_activities(ds, cfg)


## Reloading code after edits

`%autoreload 2` (cell 1) handles most edits. If you re-bind names with `from X import Y`, run this cell to refresh them too.
Reload **bottom-up** (dependencies first).

In [ ]:
import importlib
import caban.utilities, caban.sessions, caban.analysis
import caban.engram, caban.engram_sanity
import caban.population, caban.isomap, caban.epoch_analysis, caban.spatial
import caban.decoder
import caban.config, caban.loader, caban.pipeline
for _m in (caban.utilities, caban.sessions, caban.analysis,
           caban.engram, caban.engram_sanity,
           caban.population, caban.isomap, caban.epoch_analysis, caban.spatial,
           caban.decoder,
           caban.config, caban.loader, caban.pipeline):
    importlib.reload(_m)

# Re-import names that were rebound via `from X import Y`:
from caban.config import PipelineConfig
from caban.loader import load_all_mice
from caban.pipeline import (
    resolve_continuity_params, engram_idx_by_mouse, run_engram_sanity_plots,
    run_population_pca, run_population_pca_all_modes,
    run_isomap, run_epoch_pv, run_cross_session_epoch_pv,
)
print('reload complete')